# S2.1 — FastAPI App Factory\n\nVerify the app factory, lifespan, CORS, health check endpoint, and router registration.

## 1. Import and Create App

In [1]:
import sys
sys.path.insert(0, "../..")

from fastapi import FastAPI
from src.main import create_app
from src.config import Settings

app = create_app()
print(f"App type: {type(app)}")
print(f"Title:    {app.title}")
print(f"Version:  {app.version}")
print(f"Debug:    {app.debug}")
assert isinstance(app, FastAPI)
print("\n✓ create_app() returns a FastAPI instance")

App type: <class 'fastapi.applications.FastAPI'>
Title:    PaperAlchemy
Version:  0.1.0
Debug:    True

✓ create_app() returns a FastAPI instance


## 2. Custom Settings Override

In [2]:
custom_settings = Settings(app={"title": "TestApp", "version": "9.9.9", "debug": False})
custom_app = create_app(settings_override=custom_settings)

print(f"Custom title:   {custom_app.title}")
print(f"Custom version: {custom_app.version}")
print(f"Custom debug:   {custom_app.debug}")
assert custom_app.title == "TestApp"
assert custom_app.version == "9.9.9"
print("\n✓ create_app() accepts settings_override")

Custom title:   TestApp
Custom version: 9.9.9
Custom debug:   False

✓ create_app() accepts settings_override


## 3. Health Check Endpoint (GET /api/v1/ping)

In [3]:
from httpx import AsyncClient, ASGITransport

async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    resp = await client.get("/api/v1/ping")

print(f"Status: {resp.status_code}")
print(f"Body:   {resp.json()}")
assert resp.status_code == 200
assert resp.json()["status"] == "ok"
assert resp.json()["version"] == "0.1.0"
print("\n✓ GET /api/v1/ping returns 200 with correct body")

Status: 200
Body:   {'status': 'ok', 'version': '0.1.0'}

✓ GET /api/v1/ping returns 200 with correct body


## 4. PingResponse Schema Validation

In [4]:
from src.schemas.api.health import PingResponse

ping = PingResponse(**resp.json())
print(f"Schema: {ping}")
print(f"status: {ping.status}")
print(f"version: {ping.version}")
assert ping.status == "ok"
assert ping.version == "0.1.0"
print("\n✓ Response validates against PingResponse schema")

Schema: status='ok' version='0.1.0'
status: ok
version: 0.1.0

✓ Response validates against PingResponse schema


## 5. CORS Headers

In [5]:
async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    cors_resp = await client.options(
        "/api/v1/ping",
        headers={
            "Origin": "http://localhost:3000",
            "Access-Control-Request-Method": "GET",
        },
    )

print(f"Status: {cors_resp.status_code}")
print(f"Headers:")
for k, v in cors_resp.headers.items():
    if "access-control" in k:
        print(f"  {k}: {v}")

assert "access-control-allow-origin" in cors_resp.headers
print("\n✓ CORS headers present in preflight response")

Status: 200
Headers:
  access-control-allow-methods: DELETE, GET, HEAD, OPTIONS, PATCH, POST, PUT
  access-control-max-age: 600
  access-control-allow-credentials: true
  access-control-allow-origin: http://localhost:3000

✓ CORS headers present in preflight response


## 6. 404 for Unknown Routes

In [6]:
async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    not_found = await client.get("/api/v1/nonexistent")

print(f"Status: {not_found.status_code}")
assert not_found.status_code == 404
print("\n✓ Unknown routes return 404")

Status: 404

✓ Unknown routes return 404


## 7. Registered Routes Summary

In [7]:
print("Registered routes:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"  {', '.join(route.methods):8s} {route.path}")

print("\n✓ All S2.1 verifications passed!")

Registered routes:
  GET, HEAD /openapi.json
  GET, HEAD /docs
  GET, HEAD /docs/oauth2-redirect
  GET, HEAD /redoc
  GET      /api/v1/ping

✓ All S2.1 verifications passed!
